In [1]:
import sys
import os
sys.path.append(os.path.abspath('../../../'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from functions.preproc import *
from functions.ensembles import *
from functions.repeated_cv import *

from sklearn import set_config
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import make_scorer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import StratifiedKFold
from sksurv.metrics import integrated_brier_score
from sksurv.metrics import concordance_index_ipcw
from sksurv.nonparametric import kaplan_meier_estimator
from sklearn.inspection import permutation_importance

from sksurv.ensemble import RandomSurvivalForest
from sksurv.metrics import concordance_index_censored
from sksurv.util import Surv

import time
set_config(display="text")  # displays text representation of estimators

In [2]:
dataset = pd.read_csv("../../data/data_final.csv")

diagnosis_variables = []

with open("../../data/diagnosis_final.txt", "r") as file:
    for line in file:
        diagnosis_variables.append(line.strip())

treatment_variables = []
with open("../../data/treatment_final.txt", "r") as file:
    for line in file:
        treatment_variables.append(line.strip())

survival_dataset = dataset.drop('Unnamed: 0', axis=1)

feature_columns = treatment_variables + diagnosis_variables

In [3]:
# Find duplicated patients
mask = (survival_dataset.duplicated(subset='PERSON_ID', keep=False)) #& (survival_dataset['Regional LN recurrence'] == 1)

# For each duplicated PERSON_ID with recurrence, keep only the row with the lowest survival days
to_remove = (
    survival_dataset[mask]
    .sort_values(['PERSON_ID', 'Survival days'], ascending=[True, True])
    .groupby('PERSON_ID')
    .head(-1)
    .index
)

survival_dataset = survival_dataset.drop(to_remove)

In [4]:
# Prepare data
data_x = survival_dataset[feature_columns]
data_y = Surv.from_arrays(
    event=survival_dataset['Regional LN recurrence'],
    time=survival_dataset['Survival days']
)

In [5]:
sgkf = StratifiedKFold(n_splits=5, shuffle=True, random_state=12)
cv = sgkf.split(X=data_x, y=survival_dataset['Regional LN recurrence'])


train_index, test_index = next(cv)
x_train = data_x.iloc[train_index]
y_train = data_y[train_index]

x_test = data_x.iloc[test_index]
y_test = data_y[test_index]

subsampling = survival_dataset.iloc[train_index].copy()

In [6]:
param_grid = {
    'n_estimators': [100, 200, 500],    
    'max_depth': [10],    
    'max_features': ['sqrt', 'log2', 0.5],    
    'min_samples_leaf': [3, 10, 20],
    'n_controls_per_case': [1, 2, 4, 10, 15]

}
metrics = ['c-index', 'c-index-ipcw', 'int-brier-score']

In [7]:
results_df, summary_df, best_params = ensemble_repeated_grid_search(RandomSurvivalForest, x_train, y_train, param_grid, metrics,
                                  model_name='RSF Model',
                                  scale_data=False, scaler_type='standard',
                                  random_state=None, n_jobs=None,
                                  n_controls_per_case=None, dataset=subsampling,
                                  feature_columns=feature_columns,
                                  event_col='Regional LN recurrence',
                                  time_col='Survival days',
                                  n_splits=4, n_repeats=5, base_random_state=42,
                                  verbose=True)


RSF MODEL REPEATED GRID SEARCH
Model: RandomSurvivalForest
Scaling: No
Subsampling: No
Metrics: ['c-index', 'c-index-ipcw', 'int-brier-score']
Parameter Grid:
  - n_estimators: [100, 200, 500]
  - max_depth: [10]
  - max_features: ['sqrt', 'log2', 0.5]
  - min_samples_leaf: [3, 10, 20]
  - n_controls_per_case: [1, 2, 4, 10, 15]
Total hyperparameter combinations: 135
Folds per repetition: 4
Repetitions: 5
Total evaluations: 2700

[1/135] Testing params: {'n_estimators': 100, 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 3}, n_controls_per_case=1
  c-index: 0.8166 ± 0.0772
  c-index-ipcw: 0.7611 ± 0.0977
  int-brier-score: 0.0445 ± 0.0094
[2/135] Testing params: {'n_estimators': 100, 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 3}, n_controls_per_case=2
  Elapsed: 0.2min, Est. remaining: 28.4min
  c-index: 0.7989 ± 0.0788
  c-index-ipcw: 0.7281 ± 0.1332
  int-brier-score: 0.0327 ± 0.0078
[3/135] Testing params: {'n_estimators': 100, 'max_depth': 10, 'max_f

In [ ]:
summary_df.to_csv('../../results/repeated_sub/rsf_sub.csv', index=False)

: 

# Train and Test Final Model with Best Hyperparameters

In [88]:
# Load best hyperparameters from corrected_sub_results
best_results = pd.read_csv("../results/corrected_sub_results/rsf_subsampling_grid_search_results.csv")
# Select row with highest c-index-ipcw_mean
best_params = best_results.iloc[0] #.sort_values('c-index-ipcw_mean', ascending=False).iloc[0]

print("Best hyperparameters:")
print(f"  n_estimators: {int(best_params['n_estimators'])}")
max_depth_val = best_params['max_depth']
if pd.isna(max_depth_val):
    print(f"  max_depth: None")
else:
    print(f"  max_depth: {int(max_depth_val)}")
print(f"  max_features: {best_params['max_features']}")
print(f"  min_samples_leaf: {int(best_params['min_samples_leaf'])}")
print(f"  n_controls_per_case: {int(best_params['n_controls_per_case'])}")
print(f"  c-index-ipcw_mean (CV): {best_params['c-index-ipcw_mean']:.4f}")
print()

Best hyperparameters:
  n_estimators: 200
  max_depth: 5
  max_features: sqrt
  min_samples_leaf: 10
  n_controls_per_case: 2
  c-index-ipcw_mean (CV): 0.7264



In [89]:
# Prepare training fold dataframe (subsampling variable was created earlier)
train_fold = subsampling.copy()

# Apply case-control subsampling
x_train_sub, y_train_sub, sub_dataset = sample_case_control(
    train_fold=train_fold,
    event_col='Regional LN recurrence',
    time_col='Survival days',
    n_controls_per_case=int(best_params['n_controls_per_case']),
    feature_columns=feature_columns,
    random_state=12
)

print(f"Original training samples: {len(x_train)}")
print(f"Subsampled training samples: {len(x_train_sub)}")
print(f"Test samples: {len(x_test)}")

Original training samples: 2376
Subsampled training samples: 81
Test samples: 594


In [90]:
# Train final model with best hyperparameters
max_depth_val = best_params['max_depth']
max_depth = None if pd.isna(max_depth_val) else int(max_depth_val)

# Handle max_features - convert to proper type
max_features_val = best_params['max_features']
if isinstance(max_features_val, str):
    # Check if it's a numeric string
    try:
        max_features = float(max_features_val)
    except ValueError:
        # It's a string like 'sqrt' or 'log2'
        max_features = max_features_val
else:
    max_features = max_features_val

final_model = RandomSurvivalForest(
    n_estimators=int(best_params['n_estimators']),
    max_depth=max_depth,
    max_features=max_features,
    min_samples_leaf=int(best_params['min_samples_leaf']),
    random_state=12,
    n_jobs=-1
)

final_model.fit(x_train_sub, y_train_sub)
print("Model trained successfully!")

Model trained successfully!


In [91]:
# Test the model on test set
# Calculate c-index
c_index = final_model.score(x_test, y_test)

# Calculate c-index IPCW
tau = y_train_sub["time"].max()
c_index_ipcw = concordance_index_ipcw(y_train_sub, y_test, final_model.predict(x_test), tau=tau)[0]

# Calculate integrated Brier score
surv_funcs = final_model.predict_survival_function(x_test)

# Use robust time filtering
max_time = min(y_train_sub['time'].max(), y_test['time'].max()) * 0.9

# Use percentile-based times (5% to 90%)
times = np.percentile(y_train_sub['time'], np.linspace(5, 90, 10))
# Ensure times are unique and sorted
times = np.unique(times)
# Filter times that are within valid range
times = times[(times > 0) & (times <= max_time)]

preds_at_times = np.array([[fn(t) for t in times] for fn in surv_funcs])
brier_score = integrated_brier_score(y_train_sub, y_test, preds_at_times, times)

print("\\nTest Set Performance:")
print(f"  C-index: {c_index:.4f}")
print(f"  C-index IPCW: {c_index_ipcw:.4f}")
print(f"  Integrated Brier Score: {brier_score:.4f}")

\nTest Set Performance:
  C-index: 0.7938
  C-index IPCW: 0.7749
  Integrated Brier Score: 0.0220


# Feature Importance for SHAP Analysis

In [ ]:
# Extract feature importance from RSF model
# Use permutation importance since RandomSurvivalForest doesn't have feature_importances_
import pandas as pd
import numpy as np
from sklearn.inspection import permutation_importance

# IMPORTANT: Use the actual feature names from the test data
# x_test contains the actual features used
actual_features = x_test.columns.tolist()

# Calculate permutation importance on test set (using fewer repeats for speed)
print("Calculating permutation importance (this may take a moment)...")
result = permutation_importance(final_model, x_test, y_test, 
                                n_repeats=5, random_state=12, n_jobs=-1)

# Get feature importances from permutation importance
importance_df = pd.DataFrame({
    'Feature': actual_features,
    'Importance': result.importances_mean,
    'Std': result.importances_std
})

# Sort by importance (most important first)
importance_df = importance_df.sort_values('Importance', ascending=False)

# Export feature importance to CSV
importance_df.to_csv("../results/features/rsf_feature_importance.csv", index=False)

Calculating permutation importance (this may take a moment)...

Top 20 Most Important Features for RSF Model (Permutation Importance):
                                                  Feature  Importance      Std
                      Oestrogen receptor status at CB_0.0    0.046264 0.015555
                            Ki67 positive cells (%) at CB    0.030502 0.011869
                              Endocrine therapy performed    0.025077 0.007252
                                          Grade at CB_3.0    0.024258 0.010640
                      Oestrogen receptor status at CB_1.0    0.016991 0.027191
                              N (Regional nodes affected)    0.009110 0.010140
                                   Chemotherapy performed    0.008700 0.001995
                   Progesterone receptor status at CB_0.0    0.005629 0.003471
Her2 overexpression (with immunohystochemistry) at CB_1.0    0.002456 0.010232
                                                    cT_T1    0.001740 0.003

In [95]:
# Export feature importance to CSV
importance_df.to_csv("../results/features/rsf_feature_importance.csv", index=False)
print("Feature importance exported to: ../results/features/rsf_feature_importance.csv")

Feature importance exported to: ../results/features/rsf_feature_importance.csv
